# 02. Domain-Specific English Language Model Adaptation

**Course**: ICS554 Natural Language Processing · Ashesi University  
**Project**: Prosit 1 (Ankora AI Research Lab)  
**Objective**: Adapt a base English language model to a specialized domain (e.g. Healthcare, Agriculture, or Finance) and empirically demonstrate learning through perplexity reduction and generation analysis.

---
### Pipeline Overview
1. **Domain Corpus Selection**: Healthcare/clinical or agricultural domain text dataset.
2. **Base Model Benchmark**: Measuring base model zero-shot perplexity on domain test set.
3. **Domain Adaptation**: Parameter-Efficient Fine-Tuning (PEFT / LoRA) or causal LM continued pre-training.
4. **Evaluation**: Pre vs. Post adaptation perplexity and domain prompt generation comparison.

In [ ]:
import os
import sys
from pathlib import Path
import torch

# Ensure repo root is on python path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.domain_adaptation import load_model_and_tokenizer, prepare_dataset, evaluate_perplexity
from src.viz import plot_perplexity_comparison

## 1. Domain Dataset Selection
Example domain corpus: Agricultural and crop disease advisory text in Ghana/West Africa.

In [ ]:
domain_texts = [
    "Fall armyworm infests maize crops during early vegetative stages causing characteristic ragged feeding marks.",
    "Cassava mosaic disease is transmitted by the whitefly vector and causes severe leaf chlorosis and stunting.",
    "Cocoa swollen shoot virus disease results in vein clearing, red vein banding, and swollen stems in infected trees.",
    "Effective integrated pest management includes early planting, crop rotation, and biological control using parasitoid wasps.",
    "Soil nitrogen deficiency in tomato cultivation manifests as chlorosis in older lower leaves progressing upward.",
    "Drip irrigation systems optimize water delivery to root zones, reducing evaporation and fungal leaf infections.",
    "Post-harvest storage loss in grain is largely driven by maize weevil (Sitophilus zeamais) infestation.",
    "Phosphorus fertilization enhances root establishment and drought resilience in dry savanna agro-ecological zones."
]

train_split = domain_texts[:6]
test_split = domain_texts[6:]
print(f"Domain train samples: {len(train_split)}, test samples: {len(test_split)}")

## 2. Load Base Model & Baseline Perplexity

In [ ]:
BASE_MODEL_NAME = "distilgpt2"

print(f"Loading base model: {BASE_MODEL_NAME}...")
base_model, tokenizer = load_model_and_tokenizer(BASE_MODEL_NAME, use_lora=False)

test_dataset = prepare_dataset(test_split, tokenizer, max_length=128)
baseline_ppl = evaluate_perplexity(base_model, tokenizer, test_dataset)
print(f"Base Model Zero-Shot Perplexity on Domain Test Set: {baseline_ppl:.2f}")

## 3. Parameter-Efficient Fine-Tuning (LoRA Adaptation)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Setup model with LoRA adapter
adapted_model, _ = load_model_and_tokenizer(BASE_MODEL_NAME, use_lora=True, lora_r=8)
train_dataset = prepare_dataset(train_split, tokenizer, max_length=128)

training_args = TrainingArguments(
    output_dir=str(REPO_ROOT / "models" / "domain_adapted_checkpoint"),
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=5e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=adapted_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Starting LoRA Domain Adaptation Training...")
trainer.train()
print("Training completed.")

## 4. Post-Adaptation Perplexity & Generation Comparison

In [ ]:
post_ppl = evaluate_perplexity(adapted_model, tokenizer, test_dataset)
print(f"Adapted Model Perplexity on Domain Test Set: {post_ppl:.2f}")
print(f"Perplexity Improvement: {baseline_ppl - post_ppl:.2f} points")

plot_perplexity_comparison(
    ["Base Model (Zero-Shot)", "Adapted Model (LoRA)"],
    [baseline_ppl, post_ppl],
    title="Domain Adaptation Perplexity (Before vs. After)",
    save_path=REPO_ROOT / "figures" / "domain_adaptation_perplexity.png",
)

### Qualitative Comparison: Domain Prompt Generation

In [ ]:
prompt = "Cassava mosaic disease is caused by"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

with torch.no_grad():
    out_base = base_model.generate(input_ids, max_new_tokens=25, do_sample=True, temperature=0.7)
    out_adapt = adapted_model.generate(input_ids, max_new_tokens=25, do_sample=True, temperature=0.7)

print("--- Prompt ---")
print(prompt)
print("\n--- Base Model Completion ---")
print(tokenizer.decode(out_base[0], skip_special_tokens=True))
print("\n--- Adapted Model Completion ---")
print(tokenizer.decode(out_adapt[0], skip_special_tokens=True))